In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
class CategoryBinner()

In [ ]:
def cat_binning(df, cutoff):
    df_copy = df.copy()
    cat_cols = ["SubjectEyeColor", "SubjectHairColor", "VehicleStyle", "VehicleColor"]
    for col in cat_cols:
        freq = df_copy[col].value_counts(normalize=True)
        # Select all labels meeting cutoff
        labels = freq[freq > cutoff].index.tolist()
        # Filter col to include those labels + NA
        # Otherwise, miniscule labels are grouped into other
        df_copy[col] = np.where(df_copy[col].isin(labels) | df_copy[col].isna(), df_copy[col], "Other")
    return df_copy

In [ ]:
def reassign_outliers(df):
    df_copy = df.copy()
    # Potential outlier columns: VehicleYear, SubjectAge, SubjectHeight
    # Impute impossible or highly implausible values as NaN
    df_copy["VehicleYear"] = np.where((df_copy["VehicleYear"] == 1900) | (df_copy["VehicleYear"] >= 2028), np.nan, df_copy["VehicleYear"])

    df_copy["SubjectAge"] = np.where((df_copy["SubjectAge"] < 0) | (df_copy["SubjectAge"] > 119), np.nan, df_copy["SubjectAge"])

    df_copy["SubjectHeight"] = np.where((df_copy["SubjectHeight"] <= 20) | (df_copy["SubjectHeight"] >= 100), np.nan, df_copy["SubjectHeight"])

    df_copy["SubjectWeight"] = np.where((df_copy["SubjectWeight"] <= 10) | (df_copy["SubjectWeight"] >= 800), np.nan, df_copy["SubjectWeight"])

In [ ]:
preprocessing_pipeline = Pipeline([
    ("bin_categories", cat_binning(cutoff=0.01),
     "winsorize", reassign_outliers(df),
     "encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan),
     ),
])